# Montandon: Earthquakes

This notebook fetches recent earthquake data from USGS via Montandon, the global crisis data bank, filtering for magnitude, and visualizes them on a map.

In [167]:
import os
import pandas as pd
from pystac_client import Client
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
from lonboard import viz

from datetime import datetime, timedelta, timezone

## Connect to Montandon STAC

Montandon is exposed as a STAC collection, but requires authentication.

In [168]:
STAC_API_URL = "https://montandon-eoapi.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [169]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [170]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [171]:
client

<Client id=montandon-eoapi>

## Fetch most recent Events

List collections with event items, usig simple string-matching on the collection id

In [172]:
collections_list = list(client.get_collections())

In [173]:
events_collection = [c for c in collections_list if '-events' in c.id]

In [174]:
for col in events_collection:
    print(f"{col.title} | {col.id}")

DesInventar Mapped Events | desinventar-events
EM-DAT Source Events | emdat-events
GDACS Source Events | gdacs-events
GFD Source Events | gfd-events
GLIDE Source Events | glide-events
IBTrACS Source Events | ibtracs-events
IDMC GIDD Source Events | idmc-gidd-events
IDMC Internal Displacement Updates (IDU) Impacts | idmc-idu-events
IFRC Source Events | ifrcevent-events
PDC Source Events | pdc-events
USGS Events | usgs-events


### Earthquakes: Single Collection Search
Search a single collection for the most earthquakes from USGS that have a magnitude > 4 in the past week

In [175]:
now = datetime.now(timezone.utc)
one_week_ago = two_days_ago = now - timedelta(hours=168)
magnitude = 4 # minimum magnitude to search for

In [176]:
search_hazards = client.search(
    collections=["usgs-hazards"],
    datetime=f"{one_week_ago.isoformat()}/{now.isoformat()}",
    max_items=1000,
)

earthquake_hazards = [
    item for item in search_hazards.items()
    if item.properties.get("monty:hazard_detail", {}).get("severity_value", 0) > magnitude
]

In [177]:
earthquake_hazards[0]

<Item id=usgs-hazard-us7000srku-shakemap>

In [178]:
# Get the shape of an item
len(earthquake_hazards)

103

In [179]:
# ShakeMap polygons are only generated for significant events near populated areas.
# For everything else, fetch the epicenter Point from usgs-events as a fallback.
corr_ids = {item.properties['monty:corr_id'] for item in earthquake_hazards}

search_events = client.search(
    collections=["usgs-events"],
    datetime=f"{one_week_ago.isoformat()}/{now.isoformat()}",
    max_items=1000,
)

epicenters = {
    item.properties['monty:corr_id']: shape(item.geometry)
    for item in search_events.items()
    if item.properties.get('monty:corr_id') in corr_ids
}
print(f"Epicenter Points found for {len(epicenters)} / {len(corr_ids)} earthquakes")

Epicenter Points found for 88 / 88 earthquakes


In [180]:
def item_geometry(item):
    """Return ShakeMap polygon if valid, else epicenter Point from usgs-events."""
    geom = shape(item.geometry)
    if not geom.is_empty and geom.bounds != (0.0, 0.0, 0.0, 0.0):
        return geom  # ShakeMap polygon
    bbox = item.bbox
    if bbox and not all(v == 0 for v in bbox):
        return Point((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
    corr_id = item.properties.get('monty:corr_id')
    return epicenters.get(corr_id)  # epicenter Point, or None if missing from events

gdf = gpd.GeoDataFrame(
    [{
        'id': item.id,
        'monty_corr_id': item.properties['monty:corr_id'],
        'datetime': pd.to_datetime(item.properties['datetime']),
        'title': item.properties['title'],
        'magnitude': item.properties['eq:magnitude'],
        'geometry': item_geometry(item),
    } for item in earthquake_hazards],
    geometry='geometry',
    crs='EPSG:4326',
).sort_values('datetime', ascending=False)

null_geom = gdf['geometry'].isna().sum()
print(f"{len(gdf)} earthquakes total, {null_geom} with no usable geometry")

103 earthquakes total, 0 with no usable geometry


In [181]:
earthquakes = gdf
earthquakes

,id,monty_corr_id,datetime,title,magnitude,geometry
0,usgs-hazard-us7000srku-shakemap,20260608-BRA-308679-GH0311-1-GCDB,2026-06-08 19:11:21.734000+00:00,M 5.4 - southern Mid-Atlantic Ridge,5.4,"POLYGON ((-7.683 -57.467, -1.317 -57.467, -1.3..."
1,usgs-hazard-us7000srkc-shakemap,20260608-UNK-619151-GH0311-1-GCDB,2026-06-08 18:49:46.908000+00:00,"M 5.4 - 197 km SSE of Isangel, Vanuatu",5.4,"POLYGON ((168.033 -22.967, 171.9 -22.967, 171...."
2,usgs-hazard-us7000srjx-shakemap,20260608-UNK-1013875-GH0311-1-GCDB,2026-06-08 18:00:27.786000+00:00,"M 6.1 - 104 km WNW of Mantua, Cuba",6.1,"POLYGON ((-87.067 20.967, -83.15 20.967, -83.1..."
3,usgs-hazard-us7000srgz-shakemap,20260608-BRA-306879-GH0311-1-GCDB,2026-06-08 14:20:10.924000+00:00,M 5.5 - southern Mid-Atlantic Ridge,5.5,"POLYGON ((-7.7 -57.5, -1.333 -57.5, -1.333 -53..."
4,usgs-hazard-us7000srfx-shakemap,20260608-UNK-860127-GH0311-1-GCDB,2026-06-08 11:22:32.657000+00:00,"M 5.5 - 12 km WSW of Balangonan, Philippines",5.5,"POLYGON ((123.45 3.733, 127.067 3.733, 127.067..."
...,...,...,...,...,...,...
98,usgs-hazard-us7000sqtk-shakemap,20260605-UNK-640813-GH0311-1-GCDB,2026-06-05 07:54:16.110000+00:00,M 4.3 - Fiji region,4.3,POINT (-177.6208 -18.6491)
99,usgs-hazard-us7000sqtb-shakemap,20260605-UNK-522566-GH0311-1-GCDB,2026-06-05 06:54:49.152000+00:00,"M 5.1 - 103 km NNW of Villa General Roca, Arge...",5.1,POINT (-67.0453 -31.8817)
100,usgs-hazard-us7000sqrk-shakemap,20260604-UNK-895151-GH0311-1-GCDB,2026-06-04 23:41:15.399000+00:00,"M 4.8 - 32 km WSW of El Tocuyo, Venezuela",4.8,POINT (-70.0495 9.6333)
101,usgs-hazard-us7000sqrh-shakemap,20260604-UNK-807916-GH0311-1-GCDB,2026-06-04 23:28:32.786000+00:00,"M 5.2 - 77 km S of Gorontalo, Indonesia",5.2,POINT (123.0614 -0.1639)


In [182]:
print(f"There have been {len(earthquakes)} recent earthquakes with a magnitude > 5, with {len(earthquakes['monty_corr_id'].unique())} Montandon correlation IDs.")

There have been 103 recent earthquakes with a magnitude > 5, with 88 Montandon correlation IDs.


In [183]:
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.layer_extension import DataFilterExtension
import numpy as np

# Split by geometry type: ShakeMap polygons for large events, epicenter Points for the rest
poly_gdf = earthquakes[earthquakes.geometry.geom_type == 'Polygon'].copy()
point_gdf = earthquakes[earthquakes.geometry.geom_type == 'Point'].copy()

# Scale radius by magnitude — each unit doubles the circle, matching the log scale of magnitude.
# M4 → ~20 km, M5 → ~40 km, M6 → ~80 km, M6.6 → ~122 km
radii = (2 ** (point_gdf['magnitude'] - 4) * 20_000).round().astype(int).values

layers = []
if len(poly_gdf):
    layers.append(PolygonLayer.from_geopandas(
        poly_gdf,
        get_fill_color=[220, 80, 0, 120],
        get_line_color=[180, 40, 0, 220],
        get_filter_value=poly_gdf['magnitude'].values,
        extensions=[DataFilterExtension(filter_size=1)],
        filter_range=(0, 10)
    ))
if len(point_gdf):
    layers.append(ScatterplotLayer.from_geopandas(
        point_gdf,
        get_radius=radii,
        get_fill_color=[220, 80, 0, 180],
        radius_min_pixels=3,
        get_filter_value=point_gdf['magnitude'].values,
        extensions=[DataFilterExtension(filter_size=1)],
        filter_range=(0, 10)
    ))

print(f"{len(poly_gdf)} ShakeMap polygons, {len(point_gdf)} epicenter points")
m = Map(layers=layers, show_tooltip=True)

21 ShakeMap polygons, 82 epicenter points


## Manywidgets

In [184]:
from manywidgets import RangeSlider
from manywidgets.lonboard import FilterBinder
from IPython.display import display

In [185]:
slider = RangeSlider(label="Magnitude", min=0, max=10, low=2, high=10, step=1)
binders = [FilterBinder(slider, layer) for layer in layers]
display(*binders)

In [186]:
slider

In [187]:
m